# Giai đoạn 3B: Fine-tune BDD ban đêm từ base ban ngày

File này giữ tên `resume` để phù hợp cấu trúc repo cũ, nhưng vai trò hiện tại của nó là stage fine-tune ban đêm riêng biệt.
Nó không phải notebook resume lại một dry run cũ như bản legacy trước đó.

## Mục tiêu
- Nạp weight tốt nhất từ run nền ban ngày.
- Tạo một run ban đêm riêng để dễ đối chiếu metric, loss, và artifact.
- Tách stage ban đêm riêng vì môi trường hiện tại không phù hợp với flow train quá dài trong một notebook duy nhất.

## Khi nào mới chạy file này
- Sau khi `03_train_bdd100k.ipynb` đã chạy xong.
- Sau khi tồn tại `models/runs/bdd_day_base_ep50/weights/best.pt`.
- Sau khi xác nhận `data/processed/bdd_night` đã được sinh lại đầy đủ.

## Vì sao không dùng `resume=True`
- `resume=True` dùng cho trường hợp cùng một run bị ngắt giữa chừng và cần nối tiếp optimizer state, scheduler state, epoch state.
- Stage này là một run mới trên miền ban đêm, nên dùng warm-start từ `best.pt` của stage ban ngày là rõ nghĩa hơn và dễ quản lý hơn.
- Nếu run ban ngày đã hoàn tất 50/50 thì resume cùng run sẽ không đúng ngữ cảnh và dễ rơi vào logic `nothing to resume`.

## Ràng buộc máy thực tế
- Vẫn giữ mốc `50 epoch` vì máy đang train trên Windows + VSCode + RTX 3050 4 GB.
- Mục tiêu là chia chặng ngắn, checkpoint đều, để job có chết giữa chừng thì mất ít tiến độ hơn.

## Artifact và cache có thể sinh ra
- `models/runs/bdd_night_ft_from_bdd_day_ep50/`
- `labels.cache` trong cây label của `bdd_night`
- `*.npy` image cache nếu dùng `cache='disk'`

Sau khi train xong và không cần lặp lại nhanh nữa, có thể xóa cache artifact để thu hồi SSD. Không xóa weight run nếu còn cần đối chiếu.

## Bộ đếm thời gian
- Notebook có timer tổng để in thời lượng của từng chặng fine-tune sau khi `model.train()` kết thúc.


In [ ]:
import sys
import torch
import ultralytics

print(f"Python: {sys.version.split()[0]}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA build: {torch.version.cuda}")


In [ ]:
import subprocess
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM khả dụng: {vram_gb:.2f} GB")
    try:
        subprocess.run(["nvidia-smi"], check=False)
    except FileNotFoundError:
        print("nvidia-smi không có sẵn trong PATH")
else:
    print("CUDA không khả dụng; dừng trước khi fine-tune.")


## Giải thích setting fine-tune
- `batch=8`, `workers=4`, `imgsz=640`: giữ đồng bộ với stage ban ngày để sai khác chủ yếu đến từ miền dữ liệu, không phải từ setting.
- `cache='disk'`: hợp với nút thắt I/O trên máy laptop, nhưng đổi lại sẽ sinh thêm `labels.cache` và `*.npy`.
- `save_period=5`: checkpoint đều để nếu máy dừng giữa chặng thì ít tổn thất hơn.
- `resume=False`: vì đây là một run mới có ý nghĩa huấn luyện khác, không phải nối tiếp cùng experiment.

Tên run mặc định của stage này là `bdd_night_ft_from_bdd_day_ep50`. Giữ tên nhất quán để task, doc, và log không bị loạn.


In [ ]:
from pathlib import Path
import shutil
import time
import torch
from ultralytics import YOLO

BASE_DIR = Path('d:/DAT301m/proposal')
RUNS_DIR = BASE_DIR / 'models' / 'runs'
DAY_RUN_NAME = 'bdd_day_base_ep50'
NIGHT_RUN_NAME = 'bdd_night_ft_from_bdd_day_ep50'

DAY_BEST_PT = RUNS_DIR / DAY_RUN_NAME / 'weights' / 'best.pt'
NIGHT_YAML = BASE_DIR / 'data' / 'bdd_night.yaml'
NIGHT_DATA_DIR = BASE_DIR / 'data' / 'processed' / 'bdd_night'

if not torch.cuda.is_available():
    raise RuntimeError('CUDA không khả dụng, dừng trước khi fine-tune.')

if not NIGHT_YAML.exists():
    raise FileNotFoundError(f'Thiếu file YAML cấu hình: {NIGHT_YAML}')
if not NIGHT_DATA_DIR.exists():
    raise FileNotFoundError(
        f'Thư mục dữ liệu tiền xử lý bị thiếu: {NIGHT_DATA_DIR}. Chạy lại 01b_preprocess_bdd100k.ipynb trước.'
    )
if not DAY_BEST_PT.exists():
    raise FileNotFoundError(
        f'Thiếu trọng số nền ban ngày: {DAY_BEST_PT}. Chạy xong 03_train_bdd100k.ipynb trước.'
    )

RUNS_DIR.mkdir(parents=True, exist_ok=True)

start_time = time.time()
start_label = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time))
print(f'Bắt đầu fine-tune lúc: {start_label}')

model = YOLO(str(DAY_BEST_PT))
try:
    results = model.train(
        data=str(NIGHT_YAML),
        epochs=50,
        patience=15,
        batch=8,
        cache='disk',
        imgsz=640,
        device=0,
        workers=4,
        optimizer='auto',
        mosaic=0.5,
        close_mosaic=10,
        save_period=5,
        seed=0,
        project=str(RUNS_DIR),
        name=NIGHT_RUN_NAME,
        exist_ok=False,
        resume=False,
    )
finally:
    end_time = time.time()
    elapsed_seconds = end_time - start_time
    end_label = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(end_time))
    print(f'Kết thúc lúc: {end_label}')
    print(f'Thời gian chạy: {elapsed_seconds / 60:.1f} phút ({elapsed_seconds / 3600:.2f} giờ)')

run_dir = Path(model.trainer.save_dir)
best_pt = run_dir / 'weights' / 'best.pt'
tagged_pt = run_dir / 'weights' / f'{run_dir.name}.pt'

if best_pt.exists():
    shutil.copy2(best_pt, tagged_pt)
    print(f'Đã sao chép trọng số tốt nhất tới {tagged_pt}')
else:
    print(f'Không tìm thấy trọng số tốt nhất tại {best_pt}')

print('\n=== TÓM TẮT ===')
print(f'Thư mục run: {run_dir}')
print(f'Trọng số tốt nhất đã gắn nhãn: {tagged_pt}')
print('Nếu job ban đêm bị ngắt giữa chừng trước mốc 50 epoch, chỉ khi đó mới cần resume cùng run từ `last.pt` của run ban đêm.')
